In [1]:
import numpy as np 
import pandas as pd 

In [2]:
import seaborn as sns

In [3]:
%matplotlib inline

In [ ]:
sns.barplot(x='sex',y='total_bill',data=t)

In [ ]:
sns.countplot(x='sex',data=t)

In [ ]:
sns.boxplot(x='day',y='total_bill',data=t,palette='rainbow')

In [ ]:
#Can do entire dataframe with orient='h'
sns.boxplot(data=t,palette='coolwarm',orient='h')

In [ ]:
sns.boxplot(x="day",y="total_bill",
            hue="smoker",data=t, palette="coolwarm")

In [ ]:
sns.violinplot(x="day", y="total_bill", data=t,palette='rainbow')

In [ ]:
sns.violinplot(x="day",y="total_bill",data=t,hue='sex',palette='Set1')

In [7]:
import gzip
import io
import pprint
import upsetplot
from collections import defaultdict
from matplotlib_venn import venn2, venn3
from matplotlib import pyplot as plt
from urllib.request import Request, urlopen

In [8]:
def load_movie_data(sample=True):
    """
    Directly download and format data into pandas dataframe
    /!\ File is about 130Mb depending on speed connection,
    it might take some time.
    """
    req = Request('https://datasets.imdbws.com/title.basics.tsv.gz')
    req.add_header('Accept-Encoding', 'gzip')
    response = urlopen(req)
    content = gzip.decompress(response.read())
    data = pd.read_csv(io.BytesIO(content), encoding='utf8', sep="\t")
    data = data[data.isAdult == 0]
    if sample:
        # Two percent might seem low but there is approx. 7 million
        # titles without Adult category.
        return data.sample(frac=0.02)
    else:
        return data

data = load_movie_data()

/var/folders/xw/n8d_gn89383dctwyvrp_drr40000gn/T/ipykernel_11957/1251083605.py:20: DtypeWarning: Columns (4) have mixed types.Specify dtype option on import or set low_memory=False.
  data = load_movie_data()


In [ ]:
pp = pprint.PrettyPrinter(indent=4)
print("Data column names: ")
pp.pprint(list(data.columns))
print("Data shape: " + str(data.shape))
print("Genres column examples: ")
pp.pprint(data.genres.sample(5).head())


In [ ]:
# Reshape data to have for every category,
# a list of movies.
genres_movies = defaultdict(list)
for index, row in data.iterrows():
    try:
        for genre in row["genres"].split(','):
            genres_movies[genre].append(row['primaryTitle'])
    except:
        pass

pp = pprint.PrettyPrinter(indent=4, depth=1)
print("Data structure: ")
pp.pprint(genres_movies)

# Plot a simple Venn diagram and save it to file
venn2([set(genres_movies['Action']), set(genres_movies['Romance'])], set_labels = ('Action', 'Romance'))
plt.savefig("./simple_venn.png")
plt.clf()

In [ ]:
venn3([set(genres_movies['Action']), set(genres_movies['Romance']), set(genres_movies['Drama'])], set_labels = ('Action', 'Romance', 'Drama'))
plt.savefig("./large_venn.png")
plt.clf()

In [ ]:
genres_movies_set = dict()
for k, v in genres_movies.items():
    genres_movies_set[k] = set(v)

def plot_upset(genres_movies_set, movie_categories, filename):
    upset_data_sub = upsetplot.from_contents({k: v for k, v in genres_movies_set.items() if k.startswith(movie_categories)})
    upsetplot.plot(upset_data_sub)
    plt.savefig(filename)
    return

plot_upset(genres_movies_set, ('Action', 'Romance'), "./simple_upset.png")

In [ ]:
plot_upset(genres_movies_set, ('Action', 'Romance', 'Drama', 'Sci-Fi'), "./large_upset.png")